# NB01 — EDA Lanjutan, 4 Dataset

Hibah Kemdiktisaintek Part 3. Urutan dataset: **SOR -> OTDR -> HardFailure -> SoftFailure**
(tersulit ke termudah berdasar Imbalance Ratio: 558,51 -> 33,36 -> 18,62 -> 9,41).

Isi tiap dataset: cek data kosong/anomali, distribusi kelas + IR, **deteksi outlier
OTOMATIS** (analisis dan pencatatan saja, TIDAK ADA penghapusan otomatis, lihat
penjelasan di bawah), t-SNE, dan (korelasi fitur untuk dataset tabular / contoh
trace untuk SOR yang sequential).

TIDAK ADA kDN atau complexity profiling di notebook ini (eksklusif tesis CAMP-IFD,
lihat instruksi proyek).

**Data yang dipakai di notebook ini dan nb02-nb05 SAMA PERSIS** (data asli, tidak
ada baris yang dihapus di manapun secara otomatis). Deteksi outlier tetap jalan
otomatis untuk pelaporan/provenance metodologi, tapi tidak mengubah data.

**Teks di semua figure (title/axis/legend) memakai Bahasa Inggris**, karena figure
berpotensi masuk naskah IEEE Access. Narasi notebook dan output `print()` tetap
Bahasa Indonesia (untuk kerja Anda sehari-hari).

## Kenapa outlier DIANALISIS otomatis, tapi TIDAK PERNAH dihapus otomatis

Di data fault diagnosis yang timpang, **kelas minoritas SECARA DEFINISI adalah
outlier** dibanding kelas Normal. Keputusan MEMBUANG baris data adalah keputusan
berisiko, betapapun konservatif aturannya, dan TIDAK BOLEH diambil otomatis tanpa
tinjauan. Yang aman dijalankan otomatis cuma ANALISIS dan PENCATATAN:

1. Deteksi dihitung **PER KELAS** (kuartil kelasnya sendiri, bukan lintas dataset).
2. Baris yang ekstrem (>3x IQR dari kuartil kelasnya) di **>=50% fitur SEKALIGUS**
   cuma **DITANDAI sebagai kandidat**, index-nya dicatat ke CSV, TIDAK DIHAPUS.
3. Data yang dipakai nb02-nb05 SELALU data ASLI, tidak ada baris yang hilang atas
   nama "pembersihan otomatis".

Lihat `src/data_quality.py` untuk detail lengkap.

In [ ]:
import os, sys, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from collections import Counter
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
warnings.filterwarnings("ignore")

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import (DATASETS, DATASET_ORDER, TAB_DIR, FIG_DIR, PLOT_DPI,
                         FIGSIZE_STD, FIGSIZE_WIDE, CLASS_COLORS_LIST, SEED)
from src.hdmc_multiview import load_dataset, compute_ir
from src.data_quality import detect_outliers_per_class, run_outlier_analysis

sns.set_style("whitegrid")
DS_LABEL = {"sor": "SOR", "otdr": "OTDR", "hardfailure": "HardFailure", "softfailure": "SoftFailure"}
print("Urutan dataset (tersulit -> termudah):", DATASET_ORDER)

In [ ]:
### Redirect output ke results_clean/ (tidak menimpa results/ asli)
import os
RESULTS_CLEAN_ROOT = os.path.abspath(os.path.join("..", "results_clean"))
for _sub in ["tables", "figures", "models", "predictions"]:
    os.makedirs(os.path.join(RESULTS_CLEAN_ROOT, _sub), exist_ok=True)

_new_tab_dir = os.path.join(RESULTS_CLEAN_ROOT, "tables")
_new_mdl_dir = os.path.join(RESULTS_CLEAN_ROOT, "models")
_new_pred_dir = os.path.join(RESULTS_CLEAN_ROOT, "predictions")
_new_fig_dir = os.path.join(RESULTS_CLEAN_ROOT, "figures")

import src.hdmc_multiview as _m1
import src.comparator_runner as _m2
import src.experiment_runner as _m3
import src.justification_runner as _m4
import src.data_quality as _m5

for _mod in [_m1, _m2, _m3, _m4, _m5]:
    if hasattr(_mod, "TAB_DIR"):
        _mod.TAB_DIR = _new_tab_dir
    if hasattr(_mod, "MDL_DIR"):
        _mod.MDL_DIR = _new_mdl_dir
    if hasattr(_mod, "PRED_DIR"):
        _mod.PRED_DIR = _new_pred_dir

TAB_DIR = _new_tab_dir
MDL_DIR = _new_mdl_dir
PRED_DIR = _new_pred_dir
FIG_DIR = _new_fig_dir

print(f"Output diarahkan ke: {RESULTS_CLEAN_ROOT}")

## Fungsi bantu EDA

Dipakai identik untuk keempat dataset, supaya perbandingan antar-dataset apple-to-apple
(skala warna, format tabel, dan definisi metrik yang sama). Teks di figure: Bahasa Inggris.

In [ ]:
def cek_data_kosong_dan_anomali(X, y, dataset_name):
    """Cek NaN, Inf, dan baris duplikat penuh. Dataset di project ini sudah
    melalui imputasi di dataset_loaders.py (ffill/bfill/median), jadi hasil
    normal adalah NOL NaN. Fungsi ini memverifikasi itu, bukan mengasumsikan."""
    n_nan = int(np.isnan(X).sum())
    n_inf = int(np.isinf(X).sum())
    n_dup = int(pd.DataFrame(X).duplicated().sum())
    n_label_nan = int(pd.isna(y).sum())
    print(f"[{dataset_name}] shape={X.shape}  NaN={n_nan}  Inf={n_inf}  "
          f"baris_duplikat={n_dup}  label_kosong={n_label_nan}")
    if n_nan or n_inf or n_label_nan:
        print(f"  [PERINGATAN] {dataset_name} punya data kosong/anomali yang BELUM tertangani "
              f"di dataset_loaders.py, cek ulang sebelum lanjut ke eksperimen.")
    return {"dataset": dataset_name, "n_nan": n_nan, "n_inf": n_inf,
            "n_duplicate_rows": n_dup, "n_label_nan": n_label_nan}


def tabel_distribusi_dan_ir(y, class_names, dataset_name):
    cnt = Counter(y.tolist())
    n_total = len(y)
    major_n = max(cnt.values())
    rows = []
    for i, name in enumerate(class_names):
        n_i = cnt.get(i, 0)
        ir = major_n / max(n_i, 1)
        rows.append({"class_id": i, "class_name": name, "n": n_i,
                     "pct": round(n_i / n_total * 100, 2), "IR_vs_mayoritas": round(ir, 2)})
    df = pd.DataFrame(rows).sort_values("n", ascending=False)
    out_path = os.path.join(TAB_DIR, f"eda_distribusi_{dataset_name}.csv")
    df.to_csv(out_path, index=False)
    print(f"[{dataset_name}] IR keseluruhan (mayoritas/minoritas) = {compute_ir(y):.2f}")
    print(df.to_string(index=False))
    print(f"Tersimpan: {out_path}")
    return df


def plot_distribusi_kelas(df_dist, dataset_name):
    fig, ax = plt.subplots(figsize=FIGSIZE_STD)
    colors = (CLASS_COLORS_LIST * ((len(df_dist) // len(CLASS_COLORS_LIST)) + 1))[:len(df_dist)]
    bars = ax.bar(df_dist["class_name"], df_dist["n"], color=colors)
    ax.set_yscale("log")
    ax.set_ylabel("Number of Samples (log scale)")
    ax.set_xlabel("Class")
    ax.set_title(f"Class Distribution — {DS_LABEL[dataset_name]}")
    plt.xticks(rotation=30, ha="right")
    for bar, n, pct in zip(bars, df_dist["n"], df_dist["pct"]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f"{n}\n({pct}%)", ha="center", va="bottom", fontsize=8)
    plt.tight_layout()
    out_path = os.path.join(FIG_DIR, f"eda_distribusi_{dataset_name}.png")
    plt.savefig(out_path, dpi=PLOT_DPI)
    plt.show()
    print(f"Tersimpan: {out_path}")


def plot_tsne(X, y, class_names, dataset_name, max_samples=8000, seed=SEED):
    """t-SNE dibatasi max_samples (stratified subsample) untuk dataset besar
    (SOR ~100rb baris), supaya waktu komputasi wajar tanpa mengubah proporsi
    kelas (subsampling stratified, bukan potong dari awal)."""
    rng = np.random.default_rng(seed)
    if len(X) > max_samples:
        idx_sub = []
        frac = max_samples / len(X)
        for c in np.unique(y):
            c_idx = np.where(y == c)[0]
            n_take = max(1, int(len(c_idx) * frac))
            idx_sub.append(rng.choice(c_idx, size=min(n_take, len(c_idx)), replace=False))
        idx_sub = np.concatenate(idx_sub)
        X_sub, y_sub = X[idx_sub], y[idx_sub]
        print(f"[{dataset_name}] t-SNE pakai subsample stratified {len(X_sub)}/{len(X)} baris.")
    else:
        X_sub, y_sub = X, y

    X_sc = StandardScaler().fit_transform(X_sub)
    X_tsne = TSNE(n_components=2, perplexity=min(30, max(5, len(X_sub)//100)),
                  random_state=seed, init="pca").fit_transform(X_sc)

    fig, ax = plt.subplots(figsize=(10, 8))
    colors = (CLASS_COLORS_LIST * ((len(class_names) // len(CLASS_COLORS_LIST)) + 1))
    for i, name in enumerate(class_names):
        mask = y_sub == i
        if mask.sum() == 0:
            continue
        ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1], c=colors[i], label=f"{name} (n={mask.sum()})",
                   alpha=0.6, s=12)
    ax.set_xlabel("t-SNE Dimension 1")
    ax.set_ylabel("t-SNE Dimension 2")
    ax.set_title(f"t-SNE of Raw Feature Space — {DS_LABEL[dataset_name]}")
    ax.legend(loc="best", fontsize=8, markerscale=2)
    plt.tight_layout()
    out_path = os.path.join(FIG_DIR, f"eda_tsne_{dataset_name}.png")
    plt.savefig(out_path, dpi=PLOT_DPI)
    plt.show()
    print(f"Tersimpan: {out_path}")


def plot_korelasi_fitur(X, feature_names, dataset_name, max_features=40):
    """Heatmap korelasi, angka cuma ditampilkan untuk |r| > 0.5 (konsisten
    gaya EDA tesis), supaya heatmap tidak penuh angka tapi tetap menyorot
    pasangan fitur yang benar-benar berkorelasi kuat."""
    if len(feature_names) > max_features:
        print(f"[{dataset_name}] {len(feature_names)} fitur, dipotong ke {max_features} "
              f"pertama untuk keterbacaan heatmap.")
        feature_names = feature_names[:max_features]
        X = X[:, :max_features]
    df_feat = pd.DataFrame(X, columns=feature_names)
    corr = df_feat.corr()
    fig, ax = plt.subplots(figsize=FIGSIZE_WIDE)
    mask = np.triu(np.ones_like(corr, dtype=bool))
    annot = corr.copy()
    annot[abs(annot) <= 0.5] = np.nan
    annot_str = annot.map(lambda v: f"{v:.2f}" if not np.isnan(v) else "")
    sns.heatmap(corr, mask=mask, cmap="coolwarm", center=0, ax=ax,
               annot=annot_str, fmt="", cbar_kws={"label": "Pearson Correlation"})
    ax.set_title(f"Feature Correlation — {DS_LABEL[dataset_name]}")
    plt.tight_layout()
    out_path = os.path.join(FIG_DIR, f"eda_korelasi_{dataset_name}.png")
    plt.savefig(out_path, dpi=PLOT_DPI)
    plt.show()
    print(f"Tersimpan: {out_path}")


def plot_contoh_trace_per_kelas(X, y, class_names, dataset_name, n_contoh=3, seed=SEED):
    """KHUSUS SOR: pengganti heatmap korelasi. SOR adalah data window
    sequential (amplitudo bertetangga otomatis berkorelasi tinggi, heatmap
    korelasi antar-titik kurang informatif), jadi ditampilkan contoh bentuk
    trace mentah per kelas, lebih bermakna untuk data sequential."""
    rng = np.random.default_rng(seed)
    n_classes = len(class_names)
    fig, axes = plt.subplots(n_classes, 1, figsize=(10, 2.2 * n_classes), sharex=True)
    colors = (CLASS_COLORS_LIST * ((n_classes // len(CLASS_COLORS_LIST)) + 1))
    for i, name in enumerate(class_names):
        ax = axes[i] if n_classes > 1 else axes
        idx_c = np.where(y == i)[0]
        if len(idx_c) == 0:
            ax.set_title(f"{name} (no samples)")
            continue
        pick = rng.choice(idx_c, size=min(n_contoh, len(idx_c)), replace=False)
        for p in pick:
            ax.plot(X[p], alpha=0.7, color=colors[i], linewidth=1)
        ax.set_ylabel(name, fontsize=9)
        ax.set_title(f"{name} — {len(idx_c)} windows, {n_contoh} random samples", fontsize=9, loc="left")
    axes[-1].set_xlabel("Position within Window") if n_classes > 1 else ax.set_xlabel("Position within Window")
    fig.suptitle(f"Example Raw Traces per Class — {DS_LABEL[dataset_name]}", y=1.0)
    plt.tight_layout()
    out_path = os.path.join(FIG_DIR, f"eda_contoh_trace_{dataset_name}.png")
    plt.savefig(out_path, dpi=PLOT_DPI)
    plt.show()
    print(f"Tersimpan: {out_path}")


def plot_boxplot_outlier(X, y, class_names, dataset_name, feature_names=None, max_features=8):
    """Box plot per fitur per kelas, memvisualisasikan sebaran DAN whisker
    outlier (1.5x IQR standar matplotlib). Fitur dibatasi max_features
    (dipilih fitur dengan variance tertinggi) supaya tetap terbaca untuk
    dataset berfitur banyak (OTDR 32, SOR 80)."""
    n_features = X.shape[1]
    if feature_names is None:
        feature_names = [f"f{i}" for i in range(n_features)]
    if n_features > max_features:
        variances = X.var(axis=0)
        top_idx = np.argsort(variances)[::-1][:max_features]
        top_idx = np.sort(top_idx)
    else:
        top_idx = np.arange(n_features)

    n_show = len(top_idx)
    fig, axes = plt.subplots(1, n_show, figsize=(3 * n_show, 5), sharey=False)
    if n_show == 1:
        axes = [axes]
    for ax, fi in zip(axes, top_idx):
        data_per_class = [X[y == c, fi] for c in np.unique(y)]
        labels = [class_names[c] if c < len(class_names) else str(c) for c in np.unique(y)]
        bp = ax.boxplot(data_per_class, labels=labels, patch_artist=True, flierprops={"markersize": 2})
        for patch, color in zip(bp["boxes"], CLASS_COLORS_LIST):
            patch.set_facecolor(color); patch.set_alpha(0.5)
        ax.set_title(feature_names[fi], fontsize=9)
        ax.tick_params(axis="x", rotation=60, labelsize=7)
    fig.suptitle(f"Per-Feature Distribution by Class (whisker = 1.5x IQR) — {DS_LABEL[dataset_name]}", y=1.02)
    plt.tight_layout()
    out_path = os.path.join(FIG_DIR, f"eda_boxplot_outlier_{dataset_name}.png")
    plt.savefig(out_path, dpi=PLOT_DPI, bbox_inches="tight")
    plt.show()
    print(f"Tersimpan: {out_path}")


def analisis_outlier(X, y, class_names, dataset_name, feature_names=None):
    """Jalankan deteksi + PENCATATAN saja (lihat src/data_quality.py).
    TIDAK ADA penghapusan, X/y yang dipakai eksperimen tidak berubah."""
    plot_boxplot_outlier(X, y, class_names, dataset_name, feature_names=feature_names)

    _, df_report = detect_outliers_per_class(X, y, class_names)
    print(f"\n[{dataset_name}] Deteksi outlier per kelas (1,5x IQR, untuk pelaporan):")
    print(df_report.to_string(index=False))

    _, _, _, summary = run_outlier_analysis(X, y, class_names, dataset_name, verbose=True)
    return summary


eda_summary = []  # dikumpulkan tiap dataset, dipakai perbandingan akhir

---
## 1. SOR (Standard OTDR Record) — IR = 558,51, tersulit

Data lapangan hasil pengumpulan mandiri, window=80/stride=40 (TETAP, lihat config.py).
6 kelas, dominan Normal.

In [ ]:
X_sor, y_sor, class_names_sor, groups_sor = load_dataset("sor")
qc_sor = cek_data_kosong_dan_anomali(X_sor, y_sor, "sor")
print(f"Jumlah file asal (groups) unik: {len(np.unique(groups_sor))}")

In [ ]:
df_dist_sor = tabel_distribusi_dan_ir(y_sor, class_names_sor, "sor")
plot_distribusi_kelas(df_dist_sor, "sor")
eda_summary.append({"dataset": "sor", "n_samples": len(y_sor), "n_features": X_sor.shape[1],
                    "n_classes": len(class_names_sor), "IR": compute_ir(y_sor)})

In [ ]:
outlier_summary_sor = analisis_outlier(X_sor, y_sor, class_names_sor, "sor")

In [ ]:
plot_tsne(X_sor, y_sor, class_names_sor, "sor")

In [ ]:
plot_contoh_trace_per_kelas(X_sor, y_sor, class_names_sor, "sor")

---
## 2. OTDR — IR = 33,36

Basis dataset publik Abdelli dkk., dimodifikasi tidak seimbang. 8 kelas, 32 fitur
(SNR + P1-P30 + Maximum Amplitude).

In [ ]:
X_otdr, y_otdr, class_names_otdr, _ = load_dataset("otdr")
qc_otdr = cek_data_kosong_dan_anomali(X_otdr, y_otdr, "otdr")

from src.dataset_loaders import load_otdr
from src.config import OTDR_FILE
_, _, feature_names_otdr = load_otdr(OTDR_FILE, class_names_otdr)
print("Fitur:", feature_names_otdr)

In [ ]:
df_dist_otdr = tabel_distribusi_dan_ir(y_otdr, class_names_otdr, "otdr")
plot_distribusi_kelas(df_dist_otdr, "otdr")
eda_summary.append({"dataset": "otdr", "n_samples": len(y_otdr), "n_features": X_otdr.shape[1],
                    "n_classes": len(class_names_otdr), "IR": compute_ir(y_otdr)})

In [ ]:
outlier_summary_otdr = analisis_outlier(X_otdr, y_otdr, class_names_otdr, "otdr", feature_names=feature_names_otdr)

In [ ]:
plot_tsne(X_otdr, y_otdr, class_names_otdr, "otdr")

In [ ]:
plot_korelasi_fitur(X_otdr, feature_names_otdr, "otdr")

---
## 3. HardFailure — IR = 18,62

Testbed ARNO, gangguan berat (atenuasi drastis+periodik hingga daya sinyal jatuh
di bawah ambang sensitivitas penerima). Biner Normal/Failure, 12 fitur.

In [ ]:
X_hard, y_hard, class_names_hard, _ = load_dataset("hardfailure")
qc_hard = cek_data_kosong_dan_anomali(X_hard, y_hard, "hardfailure")

from src.dataset_loaders import load_failure_wide
from src.config import HARD_FILE
_, _, feature_names_hard = load_failure_wide(HARD_FILE, class_names_hard, name="HardFailure")
print("Fitur:", feature_names_hard)

In [ ]:
df_dist_hard = tabel_distribusi_dan_ir(y_hard, class_names_hard, "hardfailure")
plot_distribusi_kelas(df_dist_hard, "hardfailure")
eda_summary.append({"dataset": "hardfailure", "n_samples": len(y_hard), "n_features": X_hard.shape[1],
                    "n_classes": len(class_names_hard), "IR": compute_ir(y_hard)})

In [ ]:
outlier_summary_hard = analisis_outlier(X_hard, y_hard, class_names_hard, "hardfailure", feature_names=feature_names_hard)

In [ ]:
plot_tsne(X_hard, y_hard, class_names_hard, "hardfailure")

In [ ]:
plot_korelasi_fitur(X_hard, feature_names_hard, "hardfailure")

---
## 4. SoftFailure — IR = 9,41, termudah

Testbed ARNO, gangguan ringan (BER/OSNR terganggu tapi jaringan tetap beroperasi).
Biner Normal/Failure, 12 fitur.

In [ ]:
X_soft, y_soft, class_names_soft, _ = load_dataset("softfailure")
qc_soft = cek_data_kosong_dan_anomali(X_soft, y_soft, "softfailure")

from src.config import SOFT_FILE
_, _, feature_names_soft = load_failure_wide(SOFT_FILE, class_names_soft, name="SoftFailure")
print("Fitur:", feature_names_soft)

In [ ]:
df_dist_soft = tabel_distribusi_dan_ir(y_soft, class_names_soft, "softfailure")
plot_distribusi_kelas(df_dist_soft, "softfailure")
eda_summary.append({"dataset": "softfailure", "n_samples": len(y_soft), "n_features": X_soft.shape[1],
                    "n_classes": len(class_names_soft), "IR": compute_ir(y_soft)})

In [ ]:
outlier_summary_soft = analisis_outlier(X_soft, y_soft, class_names_soft, "softfailure", feature_names=feature_names_soft)

In [ ]:
plot_tsne(X_soft, y_soft, class_names_soft, "softfailure")

In [ ]:
plot_korelasi_fitur(X_soft, feature_names_soft, "softfailure")

---
## Ringkasan perbandingan 4 dataset

In [ ]:
df_summary = pd.DataFrame(eda_summary)
df_summary = df_summary.sort_values("IR", ascending=False)
print(df_summary.to_string(index=False))
out_path = os.path.join(TAB_DIR, "eda_ringkasan_4_dataset.csv")
df_summary.to_csv(out_path, index=False)
print(f"Tersimpan: {out_path}")

fig, ax = plt.subplots(figsize=FIGSIZE_STD)
bars = ax.bar([DS_LABEL[d] for d in df_summary["dataset"]], df_summary["IR"], color=CLASS_COLORS_LIST[:len(df_summary)])
ax.set_yscale("log")
ax.set_ylabel("Imbalance Ratio (log scale)")
ax.set_xlabel("Dataset")
ax.set_title("Imbalance Ratio Across Four Datasets (Hardest to Easiest)")
for bar, ir in zip(bars, df_summary["IR"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f"{ir:.2f}",
           ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "eda_ringkasan_ir_4_dataset.png"), dpi=PLOT_DPI)
plt.show()

# Cek kosong/anomali, ringkasan akhir
df_qc = pd.DataFrame([qc_sor, qc_otdr, qc_hard, qc_soft])
print("\nRingkasan cek data kosong/anomali:")
print(df_qc.to_string(index=False))
assert df_qc[["n_nan", "n_inf", "n_label_nan"]].sum().sum() == 0, \
    "Ada data kosong/anomali yang belum tertangani, cek dataset_loaders.py"
print("\n[OK] Tidak ada data kosong/anomali di keempat dataset.")

### Ringkasan analisis outlier, 4 dataset

Ini murni catatan/provenance (berapa baris DITANDAI sebagai kandidat kasus ekstrem
per kelas), BUKAN laporan penghapusan, karena tidak ada baris yang dihapus. Index
kandidat sudah tersimpan (`<dataset>_outlier_kandidat_index.csv`) kalau nanti
diputuskan untuk ditindaklanjuti manual.

In [ ]:
df_outlier_summary = pd.DataFrame([outlier_summary_sor, outlier_summary_otdr,
                                    outlier_summary_hard, outlier_summary_soft])
print(df_outlier_summary.to_string(index=False))
out_path = os.path.join(TAB_DIR, "eda_ringkasan_outlier_4_dataset.csv")
df_outlier_summary.to_csv(out_path, index=False)
print(f"\nTersimpan: {out_path}")
print("\nCatatan: persentase ditandai kemungkinan kecil (by design, ambang konservatif), "
      "kalau ada dataset dengan persentase besar (>1-2%), cek df_report per kelas dan "
      "kandidat_index.csv secara manual sebelum memutuskan tindakan apapun.")